In [2]:
# PyTorch main library
import torch

# Contains neural network layers like Conv2d, Linear, LayerNorm etc.
import torch.nn as nn

In [3]:
class PatchEmbedding(nn.Module):

    def __init__(
        self,
        image_size=32,      # CIFAR-10 images are 32x32
        patch_size=4,       # Split image into 4x4 patches
        in_channels=3,      # RGB image => 3 channels
        embed_dim=128       # Each patch will become a 128-dim token
    ):
        super().__init__()

        # Number of patches in one image
        # (32/4)^2 = 64 patches
        self.num_patches = (image_size // patch_size) ** 2

        # Instead of manually:
        # Patch -> Flatten -> Linear Projection
        #
        # We use Conv2D which does exactly the same thing
        # but much more efficiently.
        #
        # Input : (B, 3, 32, 32)
        # Output: (B, 128, 8, 8)
        self.projection = nn.Conv2d(
            in_channels=in_channels,
            out_channels=embed_dim,
            kernel_size=patch_size,
            stride=patch_size
        )

    def forward(self, x):

        # Input shape:
        # (B, 3, 32, 32)

        x = self.projection(x)

        # Shape becomes:
        # (B, 128, 8, 8)

        x = x.flatten(2)

        # Flatten only spatial dimensions
        # (8,8) -> 64 patches
        #
        # Shape:
        # (B, 128, 64)

        x = x.transpose(1, 2)

        # Transformer expects:
        # (Batch, Sequence Length, Embedding Dimension)
        #
        # Shape:
        # (B, 64, 128)

        return x

In [4]:
# Create a dummy batch
#
# Batch size = 2
# Channels = 3
# Height = 32
# Width = 32
x = torch.randn(2, 3, 32, 32)

# Create Patch Embedding module
patch_embed = PatchEmbedding()

# Forward pass
out = patch_embed(x)

# Expected:
# Batch size = 2
# Number of patches = 64
# Embedding dimension = 128
print(out.shape)

torch.Size([2, 64, 128])


Input Image
(2,3,32,32)

↓

After Conv
(2,128,8,8)

↓

After Flatten
(2,128,64)

↓

After Transpose
(2,64,128)

In [5]:
# class tokens + positional embeddings

class Embeddings(nn.Module):

    def __init__(
        self,
        num_patches=64,
        embed_dim=128
    ):
        super().__init__()

        # Learnable CLS token
        # Shape: (1, 1, 128)
        self.cls_token = nn.Parameter(
            torch.randn(1, 1, embed_dim)
        )

        # Learnable positional embeddings
        #
        # +1 because of CLS token
        #
        # Shape: (1, 65, 128)
        self.position_embeddings = nn.Parameter(
            torch.randn(1, num_patches + 1, embed_dim)
        )

    def forward(self, x):

        # x shape:
        # (B, 64, 128)

        batch_size = x.shape[0]

        # Copy CLS token for every image in batch
        #
        # (1,1,128) -> (B,1,128)
        cls_tokens = self.cls_token.expand(
            batch_size,
            -1,
            -1
        )

        # Add CLS token at beginning
        #
        # (B,64,128)
        #
        # becomes
        #
        # (B,65,128)
        x = torch.cat(
            [cls_tokens, x],
            dim=1
        )

        # Add positional embeddings
        #
        # Shape remains:
        # (B,65,128)
        x = x + self.position_embeddings

        return x

In [6]:
# test it
x = torch.randn(2, 3, 32, 32)

patch_embed = PatchEmbedding()

embeddings = Embeddings()

tokens = patch_embed(x)

print("After Patch Embedding:", tokens.shape)

tokens = embeddings(tokens)

print("After CLS + Position:", tokens.shape)

After Patch Embedding: torch.Size([2, 64, 128])
After CLS + Position: torch.Size([2, 65, 128])


In [7]:
# Multi-Head Self Attention (MHSA)

import torch
import torch.nn as nn


class MultiHeadSelfAttention(nn.Module):

    def __init__(
        self,
        embed_dim=128,
        num_heads=8
    ):
        super().__init__()

        # Embedding dimension must be divisible by number of heads
        assert embed_dim % num_heads == 0

        self.embed_dim = embed_dim
        self.num_heads = num_heads

        # Dimension handled by each head
        self.head_dim = embed_dim // num_heads

        # Learnable matrices for Q, K and V
        self.qkv = nn.Linear(
            embed_dim,
            embed_dim * 3
        )

        # Final projection layer
        self.projection = nn.Linear(
            embed_dim,
            embed_dim
        )

    def forward(self, x):

        # x shape:
        # (B, 65, 128)

        B, N, C = x.shape

        # Create Q, K and V together
        #
        # Shape:
        # (B, 65, 384)
        qkv = self.qkv(x)

        # Split into Q, K and V
        #
        # Shape:
        # (B, 65, 3, 8, 16)
        qkv = qkv.reshape(
            B,
            N,
            3,
            self.num_heads,
            self.head_dim
        )

        # Shape:
        # (3, B, 8, 65, 16)
        qkv = qkv.permute(
            2,
            0,
            3,
            1,
            4
        )

        # Separate Q, K and V
        #
        # Shapes:
        # (B, 8, 65, 16)
        q, k, v = qkv[0], qkv[1], qkv[2]

        # Attention scores
        #
        # (B, 8, 65, 65)
        attention_scores = (
            q @ k.transpose(-2, -1)
        )

        # Scale scores
        attention_scores = (
            attention_scores /
            (self.head_dim ** 0.5)
        )

        # Softmax
        attention_weights = torch.softmax(
            attention_scores,
            dim=-1
        )

        # Weighted sum of values
        #
        # (B, 8, 65, 16)
        attention_output = (
            attention_weights @ v
        )

        # Combine heads
        #
        # (B, 65, 8, 16)
        attention_output = (
            attention_output.transpose(1, 2)
        )

        # (B, 65, 128)
        attention_output = (
            attention_output.reshape(
                B,
                N,
                C
            )
        )

        # Final linear projection
        #
        # (B, 65, 128)
        output = self.projection(
            attention_output
        )

        return output

In [8]:
x = torch.randn(
    2,
    65,
    128
)

attention = MultiHeadSelfAttention()

out = attention(x)

print(out.shape)

torch.Size([2, 65, 128])


In [9]:
# MLP block

class MLP(nn.Module):

    def __init__(
        self,
        embed_dim=128,
        mlp_dim=512
    ):
        super().__init__()

        self.fc1 = nn.Linear(
            embed_dim,
            mlp_dim
        )

        # GELU is used in Transformers instead of ReLU
        self.gelu = nn.GELU()

        self.fc2 = nn.Linear(
            mlp_dim,
            embed_dim
        )

    def forward(self, x):

        # (B,65,128)
        x = self.fc1(x)

        # (B,65,512)
        x = self.gelu(x)

        # (B,65,128)
        x = self.fc2(x)

        return x

In [10]:
x = torch.randn(
    2,
    65,
    128
)

mlp = MLP()

out = mlp(x)

print(out.shape)

torch.Size([2, 65, 128])


In [11]:
# Transformer encoder block

class TransformerEncoderBlock(nn.Module):

    def __init__(
        self,
        embed_dim=128,
        num_heads=8,
        mlp_dim=512
    ):
        super().__init__()

        # LayerNorm before Attention
        self.norm1 = nn.LayerNorm(
            embed_dim
        )

        self.attention = MultiHeadSelfAttention(
            embed_dim=embed_dim,
            num_heads=num_heads
        )

        # LayerNorm before MLP
        self.norm2 = nn.LayerNorm(
            embed_dim
        )

        self.mlp = MLP(
            embed_dim=embed_dim,
            mlp_dim=mlp_dim
        )

    def forward(self, x):

        # ===== Attention Block =====

        x = x + self.attention(
            self.norm1(x)
        )

        # ===== MLP Block =====

        x = x + self.mlp(
            self.norm2(x)
        )

        return x

In [12]:
x = torch.randn(
    2,
    65,
    128
)

encoder = TransformerEncoderBlock()

out = encoder(x)

print(out.shape)

torch.Size([2, 65, 128])


In [71]:
# Vision transformer

class VisionTransformer(nn.Module):

    def __init__(
        self,
        image_size=32,
        patch_size=4,
        in_channels=3,
        num_classes=10,
        embed_dim=128,
        depth=6,
        num_heads=8,
        mlp_dim=512
    ):
        super().__init__()

        # Convert image into patch tokens
        self.patch_embedding = PatchEmbedding(
            image_size=image_size,
            patch_size=patch_size,
            in_channels=in_channels,
            embed_dim=embed_dim
        )

        num_patches = self.patch_embedding.num_patches

        # Add CLS token and positional embeddings
        self.embedding_layer = Embeddings(
            num_patches=num_patches,
            embed_dim=embed_dim
        )

        # Stack multiple Transformer Encoder Blocks
        self.encoder = nn.Sequential(
            *[
                TransformerEncoderBlock(
                    embed_dim=embed_dim,
                    num_heads=num_heads,
                    mlp_dim=mlp_dim
                )
                for _ in range(depth)
            ]
        )

        # Final normalization
        self.norm = nn.LayerNorm(
            embed_dim
        )

        # Classification head
        self.head = nn.Linear(
            embed_dim,
            num_classes
        )

    def forward(self, x):

        # (B,3,32,32)
        x = self.patch_embedding(x)

        # (B,64,128)
        x = self.embedding_layer(x)

        # (B,65,128)
        x = self.encoder(x)

        # Take CLS token only
        #
        # Shape:
        # (B,128)
        cls_token = x[:, 0]

        cls_token = self.norm(
            cls_token
        )

        # (B,10)
        logits = self.head(
            cls_token
        )

        return logits

In [72]:
x = torch.randn(
    2,
    3,
    32,
    32
)

model = VisionTransformer()

out = model(x)

print(out.shape)

torch.Size([2, 10])


In [73]:
# seeing what actually is being built (not in the workflow), checking if the model trains comfortably on colab

model = VisionTransformer()

total_params = sum(
    p.numel() for p in model.parameters()
)

print(f"Total Parameters: {total_params:,}")

Total Parameters: 1,205,898


In [74]:
# CIFAR-10 dataset
import torchvision

# Image transformations
import torchvision.transforms as transforms

# Creates batches
from torch.utils.data import DataLoader

In [75]:
transform_train = transforms.Compose([

    transforms.RandomCrop(
        32,
        padding=4
    ),

    transforms.RandomHorizontalFlip(),

    transforms.ToTensor(),

    transforms.Normalize(
        (0.5, 0.5, 0.5),
        (0.5, 0.5, 0.5)
    )

])

transform_test = transforms.Compose([

    transforms.ToTensor(),

    transforms.Normalize(
        (0.5, 0.5, 0.5),
        (0.5, 0.5, 0.5)
    )

])

In [76]:
train_dataset = torchvision.datasets.CIFAR10(
    root="./data",
    train=True,
    download=True,
    transform=transform_train
)

test_dataset = torchvision.datasets.CIFAR10(
    root="./data",
    train=False,
    download=True,
    transform=transform_test
)

In [77]:
train_loader = DataLoader(
    train_dataset,
    batch_size=128,
    shuffle=True
)

test_loader = DataLoader(
    test_dataset,
    batch_size=128,
    shuffle=False
)

In [78]:
print("Training Images:", len(train_dataset))
print("Test Images:", len(test_dataset))

Training Images: 50000
Test Images: 10000


In [79]:
# Check whether GPU is available

device = torch.device(
    "cuda" if torch.cuda.is_available()
    else "cpu"
)

print("Device:", device)

Device: cuda


In [80]:
model = VisionTransformer()

# Move model to GPU
model = model.to(device)

print(model)

VisionTransformer(
  (patch_embedding): PatchEmbedding(
    (projection): Conv2d(3, 128, kernel_size=(4, 4), stride=(4, 4))
  )
  (embedding_layer): Embeddings()
  (encoder): Sequential(
    (0): TransformerEncoderBlock(
      (norm1): LayerNorm((128,), eps=1e-05, elementwise_affine=True)
      (attention): MultiHeadSelfAttention(
        (qkv): Linear(in_features=128, out_features=384, bias=True)
        (projection): Linear(in_features=128, out_features=128, bias=True)
      )
      (norm2): LayerNorm((128,), eps=1e-05, elementwise_affine=True)
      (mlp): MLP(
        (fc1): Linear(in_features=128, out_features=512, bias=True)
        (gelu): GELU(approximate='none')
        (fc2): Linear(in_features=512, out_features=128, bias=True)
      )
    )
    (1): TransformerEncoderBlock(
      (norm1): LayerNorm((128,), eps=1e-05, elementwise_affine=True)
      (attention): MultiHeadSelfAttention(
        (qkv): Linear(in_features=128, out_features=384, bias=True)
        (projection): Li

In [81]:
# Multi-class classification

criterion = nn.CrossEntropyLoss()

In [82]:
optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=3e-4,
    weight_decay=0.05
)

In [83]:
def train_one_epoch(
    model,
    dataloader,
    criterion,
    optimizer,
    device
):

    model.train()

    running_loss = 0.0
    correct = 0
    total = 0

    for images, labels in dataloader:

        images = images.to(device)
        labels = labels.to(device)

        optimizer.zero_grad()

        outputs = model(images)

        loss = criterion(
            outputs,
            labels
        )

        loss.backward()

        optimizer.step()

        running_loss += loss.item()

        _, predicted = outputs.max(1)

        total += labels.size(0)

        correct += predicted.eq(labels).sum().item()

    epoch_loss = (
        running_loss /
        len(dataloader)
    )

    epoch_accuracy = (
        100 * correct / total
    )

    return epoch_loss, epoch_accuracy

In [84]:
@torch.no_grad()
def evaluate(
    model,
    dataloader,
    criterion,
    device
):

    model.eval()

    running_loss = 0.0
    correct = 0
    total = 0

    for images, labels in dataloader:

        images = images.to(device)
        labels = labels.to(device)

        outputs = model(images)

        loss = criterion(
            outputs,
            labels
        )

        running_loss += loss.item()

        _, predicted = outputs.max(1)

        total += labels.size(0)

        correct += predicted.eq(labels).sum().item()

    epoch_loss = (
        running_loss /
        len(dataloader)
    )

    epoch_accuracy = (
        100 * correct / total
    )

    return epoch_loss, epoch_accuracy

In [85]:
num_epochs = 20

for epoch in range(num_epochs):

    train_loss, train_acc = train_one_epoch(
        model,
        train_loader,
        criterion,
        optimizer,
        device
    )

    test_loss, test_acc = evaluate(
        model,
        test_loader,
        criterion,
        device
    )

    print(
        f"Epoch [{epoch+1}/{num_epochs}] "
        f"Train Loss: {train_loss:.4f} "
        f"Train Acc: {train_acc:.2f}% "
        f"Test Acc: {test_acc:.2f}%"
    )

Epoch [1/20] Train Loss: 1.9171 Train Acc: 29.05% Test Acc: 42.06%
Epoch [2/20] Train Loss: 1.6021 Train Acc: 42.01% Test Acc: 47.11%
Epoch [3/20] Train Loss: 1.4577 Train Acc: 47.26% Test Acc: 51.13%
Epoch [4/20] Train Loss: 1.3688 Train Acc: 50.51% Test Acc: 53.56%
Epoch [5/20] Train Loss: 1.2987 Train Acc: 53.01% Test Acc: 56.37%
Epoch [6/20] Train Loss: 1.2397 Train Acc: 55.23% Test Acc: 57.68%
Epoch [7/20] Train Loss: 1.1955 Train Acc: 56.77% Test Acc: 58.28%
Epoch [8/20] Train Loss: 1.1535 Train Acc: 58.63% Test Acc: 60.45%
Epoch [9/20] Train Loss: 1.1070 Train Acc: 60.16% Test Acc: 61.20%
Epoch [10/20] Train Loss: 1.0711 Train Acc: 61.77% Test Acc: 62.12%
Epoch [11/20] Train Loss: 1.0382 Train Acc: 62.89% Test Acc: 63.70%
Epoch [12/20] Train Loss: 1.0054 Train Acc: 64.06% Test Acc: 64.70%
Epoch [13/20] Train Loss: 0.9779 Train Acc: 65.11% Test Acc: 64.19%
Epoch [14/20] Train Loss: 0.9479 Train Acc: 66.19% Test Acc: 65.96%
Epoch [15/20] Train Loss: 0.9252 Train Acc: 66.76% Test A